In [1]:
import numpy as np
import torch

In [2]:
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])
print(X)
print(y)

[160 170 180 190]
[0 0 1 1]


In [3]:
X_mean = np.mean(X)
X_std = np.std(X)
X_norm = (X - X_mean) / X_std
print(X_mean)
print(X_std)
print(X_norm)

175.0
11.180339887498949
[-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print(X_norm_tensor)
print(y_tensor)

print(X_norm_tensor.shape)
print(y_tensor.shape)

tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
tensor([[0.],
        [0.],
        [1.],
        [1.]])
torch.Size([4, 1])
torch.Size([4, 1])


In [5]:
class PerceptronModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(1, 1)
    
    def forward(self, x):
        H = self.linear(x)
        # 이번 파일은 torch.nn.BCEWithLogitsLoss()를 쓰므로
        # sigmoid를 적용하지 않고 H를 그대로 반환
        return H

In [6]:
torch.manual_seed(42)
model = PerceptronModel()

print(model.linear.weight)
print(model.linear.bias)
print(model)

Parameter containing:
tensor([[0.7645]], requires_grad=True)
Parameter containing:
tensor([0.8300], requires_grad=True)
PerceptronModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


In [7]:
# 이번 파일의 model은 H를 반환합니다. 그래서 예측 확률을 보려면 직접 sigmoid를 적용해야 합니다
# 아직 학습 전이라 weight, bias는 랜덤 초기값 상태 (예측이 맞을 필요는 없음)
with torch.no_grad():
    H_test = model(X_norm_tensor)   # 선형 계산값 H
    z_test = torch.sigmoid(H_test)  # sigmoid를 적용한 예측 확률 (=기존 수식의 z)

# H와 z(확률)는 둘 다 각 데이터에 대해 하나씩 나오므로 (4, 1) shape이어야 합니다.
print(H_test.shape)
print(H_test)
print(z_test.shape)
print(z_test)

torch.Size([4, 1])
tensor([[-0.1957],
        [ 0.4881],
        [ 1.1719],
        [ 1.8557]])
torch.Size([4, 1])
tensor([[0.4512],
        [0.6197],
        [0.7635],
        [0.8648]])


In [8]:
learning_rate = 0.1
epochs = 1000

# torch.nn.BCEWithLogitsLoss()
# - sigmoid와 BCE Cost 계산을 내부에서 함께 처리하는 부품
# - 사용법 : Cost = criterion(H, y_tesnsor) <- z가 아니라 H를 넣음
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

print(learning_rate)
print(epochs)
print(criterion)
# 이 출력 결과에 보이는 값들이 optimizer가 업데이트할 학습 대상입니다.
print(list(model.parameters()))

0.1
1000
BCEWithLogitsLoss()
[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [9]:
# BCEWithLogitsLoss로 경사 하강법 학습 (이번 실습의 핵심 루프)

for epoch in range(epochs):
    # 이전에 계산된 기울기 초기화
    optimizer.zero_grad()

    # model호출하면 내부적으로 forward가 실행됨 (이번 실습에선 H를 반환)
    H = model(X_norm_tensor)

    # BCEWithLogitsLoss는 H를 입력으로 받음 (sigmoid는 내부에서 처리)
    # 주의: 여기서 z = torch.sigmoid(H)를 직접 만들지 않음 (그건 BCELoss 방식)
    mean_cost = criterion(H, y_tensor)

    # model 내부 파라미터의 gradient 자동계산
    mean_cost.backward()

    # optimizer가 model 내부 weight와 bias 업데이트
    optimizer.step()

    # 학습상태 출력
    if epoch % 100 == 0 or epoch == epochs -1:
        print(
            f"epoch={epoch}, "
            f"Cost={mean_cost.item():.6f}, "
            f"weight(a)={model.linear.weight.item():.6f}, "
            f"bias(b)={model.linear.bias.item():.6f}"
        )
    
    if epoch < 3:
        print(
            f"(확인용) model.linear.weight.grad={model.linear.weight.grad.item():.6f}, "
            f"model.linear.bias.grad={model.linear.bias.grad.item():.6f}"
        )

epoch=0, Cost=0.495464, weight(a)=0.793780, bias(b)=0.812529
(확인용) model.linear.weight.grad=-0.292415, model.linear.bias.grad=0.174793
(확인용) model.linear.weight.grad=-0.286153, model.linear.bias.grad=0.169918
(확인용) model.linear.weight.grad=-0.280072, model.linear.bias.grad=0.165171
epoch=100, Cost=0.178670, weight(a)=2.290082, bias(b)=0.173212
epoch=200, Cost=0.125357, weight(a)=3.002210, bias(b)=0.061586
epoch=300, Cost=0.099283, weight(a)=3.509002, bias(b)=0.026837
epoch=400, Cost=0.082901, weight(a)=3.912263, bias(b)=0.013229
epoch=500, Cost=0.071398, weight(a)=4.250606, bias(b)=0.007116
epoch=600, Cost=0.062789, weight(a)=4.543496, bias(b)=0.004091
epoch=700, Cost=0.056068, weight(a)=4.802371, bias(b)=0.002480
epoch=800, Cost=0.050660, weight(a)=5.034644, bias(b)=0.001570
epoch=900, Cost=0.046207, weight(a)=5.245449, bias(b)=0.001031
epoch=999, Cost=0.042507, weight(a)=5.436657, bias(b)=0.000701


In [10]:
# 입력 특성이 1개라 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
print("학습된 weight(a):", model.linear.weight.item())
print("학습된 bias(b):", model.linear.bias.item())

# tensor 원본 형태도 함께 확인해 둡니다. (shape과 requires_grad 표시를 볼 수 있습니다.)
print("model.linear.weight:", model.linear.weight)
print("model.linear.bias:", model.linear.bias)

# 학습 후 grad도 확인해 봅니다. (마지막 epoch의 grad가 남아 있습니다.)
#   기존 grad_a -> model.linear.weight.grad
#   기존 grad_b -> model.linear.bias.grad
print("model.linear.weight.grad:", model.linear.weight.grad)
print("model.linear.bias.grad:", model.linear.bias.grad)

학습된 weight(a): 5.436657428741455
학습된 bias(b): 0.0007013113354332745
model.linear.weight: Parameter containing:
tensor([[5.4367]], requires_grad=True)
model.linear.bias: Parameter containing:
tensor([0.0007], requires_grad=True)
model.linear.weight.grad: tensor([[-0.0185]])
model.linear.bias.grad: tensor([2.6390e-05])


In [13]:
# 키가 185cm인 사람이 농구선수인지 예측합니다.
input_height = 185

# 새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 합니다.
# 학습 때 사용한 X_mean, X_std 를 그대로 다시 사용합니다. (새로 계산하면 안 됩니다.)
input_norm = (input_height - X_mean) / X_std

# 예측은 학습이 아니므로 weight, bias를 업데이트하지 않습니다.
# 따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산합니다.
with torch.no_grad():
    # torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 합니다.
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print("input_norm_tensor shape:", input_norm_tensor.shape)

    # model은 H를 반환합니다. (선형 계산값 - 확률 아님)
    H_new = model(input_norm_tensor)

    # 예측 확률을 보려면 H에 sigmoid를 직접 적용합니다.
    # probability = sigmoid(H_new) 이며, 이것이 기존 수식의 z와 같은 의미입니다.
    probability = torch.sigmoid(H_new)

    # 0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
    pred = 1 if probability.item() >= 0.5 else 0

# H_new       : 선형 계산값
# probability : sigmoid(H_new)로 계산한 예측 확률 (= 기존 수식의 z)
# pred        : probability가 0.5 이상인지에 따라 정한 최종 분류 결과
print("H_new:", H_new.item())
print("probability:", probability.item())
print("prediction:", pred)

print(f"\n키가 {input_height}cm인 사람의 농구선수 확률(probability): {probability.item():.4f}")
if pred == 1:
    print("판별 결과: 농구선수입니다.")
else:
    print("판별 결과: 농구선수가 아닙니다.")

input_norm_tensor shape: torch.Size([1, 1])
H_new: 4.863395690917969
probability: 0.9923350214958191
prediction: 1

키가 185cm인 사람의 농구선수 확률(probability): 0.9923
판별 결과: 농구선수입니다.
